# 04 - Multi-Label Category Classification

**Goal:** predict one *or more* categories for a single piece of feedback.

> "The app is very slow and payment keeps failing."
> -> `["performance", "payment"]`

We use:
- **MultiLabelBinarizer** - turns a list of label-sets into a binary matrix
- **OneVsRestClassifier** - one binary classifier per category
- **Logistic Regression** as the base classifier

**Dataset:** the small, **manually labeled** supplement (`data/processed/manual_categories.csv`). The airline dataset does not have reliable app-category multi-labels, so this supplement is clearly documented as manually labeled (see README).

> NOTE: This notebook also demonstrates the **airline complaint categories** as a single-label sanity check in the bonus section.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np

from src.data_loader import load_manual_multilabel_dataset
from src.multilabel_classifier import (
    prepare_multilabel_data,
    train_multilabel_model,
    predict_multilabel_with_fallback,
)
from src.evaluation import evaluate_multilabel

In [ ]:
df = load_manual_multilabel_dataset()
print("Rows:", len(df))

# Count label occurrences
from collections import Counter
counts = Counter()
for cats in df["categories"].str.split(","):
    counts.update(cats)
print("Label occurrences:", dict(counts))

multi = df["categories"].str.contains(",").sum()
print(f"Multi-label rows: {multi} ({multi/len(df)*100:.0f}%)")
df.head()

In [ ]:
X_train, X_test, y_train, y_test, mlb = prepare_multilabel_data(df)
print("Classes:", list(mlb.classes_))
print("Train shape:", y_train.shape, "| Test shape:", y_test.shape)

## The multi-label trick

`MultiLabelBinarizer` converts:

```
["performance", "payment"]  ->  [0,0,1,0,1,1,0,0]   (one row per label)
["support"]                 ->  [0,0,0,1,0,0,0,0]
```

`OneVsRestClassifier` then trains **one binary classifier per column** — e.g. "is this about payment? yes/no".

In [ ]:
model = train_multilabel_model(X_train, y_train)
y_pred_bin = model.predict(X_test)

results = evaluate_multilabel(
    y_test, y_pred_bin,
    labels=list(mlb.classes_),
    task_name="Multi-Label Category",
)

## Interpreting the metrics

- **Micro F1**: counts each label decision, giving equal weight to every (sample, category) pair. Best for imbalanced data.
- **Macro F1**: averages F1 over categories, treating each category equally.
- **Hamming loss**: fraction of labels that are wrong. Lower is better.

Because the manual supplement is small (a few hundred rows), these numbers should be read as a *demonstration* of the technique rather than a production-quality benchmark.

## Test the key multi-label example

In [ ]:
tests = [
    "The application is very slow and payment keeps failing.",
    "The support team solved my issue very quickly.",
    "Please add a wishlist feature.",
    "I cannot log in to my account since yesterday.",
    "The new dashboard is beautiful but the charts are slow to render.",
]

for t in tests:
    preds = predict_multilabel_with_fallback(model, [t], mlb)[0]
    print(f"{preds}  |  {t[:55]}")

The flagship example correctly returns **both** `performance` and `payment` — it was NOT forced into a single category.

## Bonus: single-label airline complaint categories (sanity check)

As a quick single-label demo, we train the same TF-IDF + LogisticRegression pipeline on the airline `negativereason`-derived categories (`feedback_sentiment_cat.csv`).

In [ ]:
single = pd.read_csv("../data/processed/feedback_sentiment_cat.csv")
single["category"] = single["category"].str.lower()
print(single["category"].value_counts())

from src.category_classifier import prepare_category_data, train_category_model
from src.evaluation import evaluate_single_label

Xtr, Xte, ytr, yte = prepare_category_data(single)
cat_model = train_category_model(Xtr, ytr)
cat_pred = cat_model.predict(Xte)
cat_results = evaluate_single_label(
    yte, cat_pred,
    labels=sorted(set(yte)),
    task_name="Single-label Category",
)